# 00 — Configuración de entorno Hugging Face / Colab GPU (TFM-21)

Notebook de arranque para Google Colab con GPU. Verifica runtime, autenticación Hugging Face, descarga de modelos base (mBERT y XLM-R) y ejecuta un smoke test del pipeline de fine-tuning del proyecto.

**Requisitos en Colab:** Runtime → Cambiar tipo de entorno de ejecución → **T4 GPU** (o superior).

In [ ]:
# 1) Verificar GPU disponible
!nvidia-smi

import torch

print("torch:", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("dispositivo:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: activa GPU en Runtime > Cambiar tipo de entorno de ejecución")

In [ ]:
# 2) Clonar repositorio y entrar al directorio del proyecto
REPO_URL = "https://github.com/caeher/cde-project-ml-2026.git"
PROJECT_DIR = "/content/cde-project-ml-2026"

import os
from pathlib import Path

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git rev-parse --short HEAD

In [ ]:
# 3) Instalar dependencias del proyecto
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# 4) Autenticación Hugging Face (necesaria para descargar modelos base)
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# 5) Verificación reproducible del entorno HF (TFM-21)
!python scripts/verify_hf_env.py

In [ ]:
# 6) Smoke test: fine-tuning ligero con GPU (demostración)
# Usa subset pequeño para validar el pipeline sin largos tiempos de entrenamiento.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

from discurso_odio.data.loader import load_dataset, prepare_dataset
from discurso_odio.features.preprocessing import preprocess_dataframe
from discurso_odio.models.transformers_ft import fine_tune_transformer

train_df = preprocess_dataframe(prepare_dataset(load_dataset("entrenamiento")))
X = train_df["texto_limpio"]
y = train_df["label"]

result = fine_tune_transformer(
    X,
    y,
    model_key="mbert",
    epochs=1,
    sample_size=80,
    max_length=64,
)
print("Checkpoint:", result["output_dir"])
print("Métricas:", result["metrics_path"])
print("Eval:", {k: v for k, v in result["eval_metrics"].items() if k.startswith("eval_")})